In [1]:
from wizard import Node, Flow, InputField, RadioField, GoNext

In [2]:
profile_screen = Node(
    "profile_screen",
    "Profile",
    instructions="""Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.""",
    fields=[
        InputField("tenant_name", "Tenant Name", "fills the name of the calling user/tenant"),
        InputField("tenant_address", "Tenant Address", "fills the address of the user/tenant")
    ],
    next="location_screen"
)

location_screen = Node(
    "location_screen",
    "Location",
    instructions="""Learn whether the issue is inside their home or outside""",
    fields=[
        RadioField("location", "Location", "Select the location", [
            "Inside home",
            "Outside home"
        ])
    ],
    next=lambda ctx: "inside_home_area_screen" if ctx["location"] == "Inside home" else "outside_home_area_screen"
)


inside_home_area_screen = Node(
    "inside_home_area_screen",
    "Inside Home Area",
    instructions="Ask the user about the area where the issue is",
    fields=[
        RadioField("area", "Area", "", [
            "Floors, Walls, Ceilings and Stairs",
            "Plumbing",
            "Doors, Locks and Windows",
            "Electrics",
            "Alarms & Door Entry",
            "Heating & Hot Water",
            "Empty Repair"
        ])
    ],
    next={
        "Floors, Walls, Ceilings and Stairs": "floors_walls_stairs_screen",
        "Plumbing": "plumbing_screen",
        # "Roof leaking": "roof_leaking_screen"
    }
)

floors_walls_stairs_screen = Node(
    "floors_walls_stairs_screen",
    "Area: Floors, walls and Stairs",
    "Ask the user about the precise area where the issue is",
    fields=[
        RadioField(
            "area_tier_2",
            "Which area excatly has the problem?",
            "",
            [
                "Floors",
                "Ceilings",
                "Walls",
                "Stairs"
            ]
        )
    ],
    next={
        "Ceilings": "ceilings_issues_screen",
        "Floors": "floor_issues_screen"
    }
)

ceiling_issues_screen = Node(
    "ceilings_issues_screen",
    "Ceilings Issues",
    "Learn from the user which kind of ceiling issue they area dealing with",
    fields=[
        RadioField("ceiling_issue", "Ceiling Issue", "", [
            "Ceiling is falling down",
            "Cracks in the ceiling",
            "Roof leaking"
    ])
    ],
    next={
        "Ceiling is falling down": "ceiling_is_falling_down_screen",
        "Cracks in the ceiling": "cracks_in_the_ceiling_screen",
        "Roof leaking": "roof_leaking_screen"
    }
)

cracks_in_the_ceiling_screen = Node(
    "cracks_in_the_ceiling_screen",
    "Cracks in the ceiling",
    "Learn whether the crack can fit a 1 euro coin or not",
    fields=[
        RadioField("crack_can_fit_coin", "Could you fit a one euro coin in the gap?", "", 
                   [
                       "yes",
                       "no"
                   ])
    ],
    next="exact_location_screen"
)

In [3]:
# flow = Flow([profile_screen, 
#              location_screen, 
#              inside_home_area_screen, 
#              ceiling_issues_screen, 
#              floors_walls_stairs_screen, 
#              ceiling_issues_screen, 
#              cracks_in_the_ceiling_screen])

# flow.play_actions(flow.current_action_model()())
# flow.play_actions([flow.current_action_model().__args__[0](value="Inside home")])
# flow.play_actions([GoNext()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Floors, Walls and Stairs")])
# flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Ceilings")])
# flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Cracks in the ceiling")])
# flow.play_actions([Next()])




# print(flow.render())

In [ ]:
import warnings
warnings.filterwarnings("ignore")       

import os
from typing import Optional

from wizard import InputField, RadioField, GoNext, GoBack

import openai
from pydantic import BaseModel, Field

SYS = """
You are a customer support AI agent operating within our agent scripting system. Your role is to assist customers with their inquiries, issues, and requests in a helpful, professional, and efficient manner.

You will be given two pieces of information, event stream and agent script.

<event_stream>
The event stream contains all the events that lead to the current state, including the conversation you have with the user, and the actions you took.
User messages will show in the following format:
[User: ...]
Agent (you) messages will show in the following format:
[Agent: ...]
Actions you took will show in the following format:
[Agent took action: ...]
</event_stream>

<agent_script>
contains the current node of the agent script, you can use the different actions to navigate across nodes or modify/input required data.
</agent_script>

<actions>
You are given actions that you can perform on the screen, you are also given actions to navigate nodes
Once you are done with a node, you can use the 'GoNext' action to proceeed
</actions>

<response-rules>
Do not be too chatty, you should not response after each single action, you should talk to the user when needed.
The user should not "feel" like you are actually filling out some kind of form, but rather, give them the illusion that you are having a seamless conversation with them.
</response-rules>

<agent-script-rules>
You should make proper use of the actions provided
You can use the GoBack action to go to a previous node if you need to fix or change some data in previous nodes
</agent-script-rules>

<general-rules>
- Each node comes with an instruction that you should follow
- Never provide inputs that the user has not provided!
- Take one action at a time
</general-rules>
""".strip()

SYS_O3 = """
You are a customer support AI agent operating within our agent scripting system. Your role is to assist customers with their inquiries, issues, and requests in a helpful, professional, and efficient manner.

You will be given two pieces of information, event stream and agent script.

<event_stream>
The event stream contains all the events that lead to the current state, including the conversation you have with the user, and the actions you took.
User messages will show in the following format:
[User: ...]
Agent (you) messages will show in the following format:
[Agent: ...]
Actions you took will show in the following format:
[Agent took action: ...]
</event_stream>

<agent_script>
Contains the current node of the agent script; you can use the different actions to navigate across nodes or modify/input required data.
</agent_script>

<actions>
You are given actions that you can perform on-screen; you are also given actions to navigate nodes.
Once you are done with a node, use the 'GoNext' action to proceed.
</actions>

<action-policy>
- **Single-action rule (STRICT).**
  • If you take an action this turn, supply **exactly one** non-null action field.  
  • All other action fields must be omitted or `null`.

- **No-action turns.**
  • If no action is required, output `action=None` (do NOT send an ActionModel full of nulls).

- **Node-completion rule.**
  • **When every mandatory field in the current node is populated, your very next action must be `GoNext`.**  
  • Do **NOT** re-fill or re-select a field that already has a value; that will be treated as a duplicate action.

- After performing an action, send a user message only if it adds value (confirms a change or requests new data).
</action-policy>

<input-parsing-rules>
- **Extract before you ask.** Each time a user message arrives, scan it (and, if helpful, recent user messages in the <event_stream>) for any information that fills the current node’s missing fields.
- If you can confidently map user-supplied information to a field, **fill the field immediately** (single-action rule still applies) and send a **brief confirmation** such as  
- Only ask a question for data that is still missing **after** this extraction step.
- When user language is ambiguous, make a best match, then confirm with the user.
</input-parsing-rules>

<response-rules>
- Be concise (≤ 35 words); phrase naturally.
- **No premature wrap-ups.** Do **NOT** use closing lines like “How can I help you today?” or “Is there anything else?” unless the script is at its terminal node (no further `GoNext`).
- Avoid repeating confirmed information unless it changes or the user asks.
- End with a clear prompt for the next required piece of information when appropriate.
</response-rules>

<agent-script-rules>
- Make proper use of actions.
- Use **GoBack** only when needed; after fixing data, immediately `GoNext`—without re-asking satisfied prompts.
- **No duplicate actions.** Never fill/select the same field with the same value more than once.  
  Attempting to re-fill an already-filled field is an error; instead, proceed with `GoNext`.
</agent-script-rules>

<error-handling-rules>
- If the user asks to amend previous data, acknowledge briefly, `GoBack`, update the field, then `GoNext`—no extra chatter.
- If user input does not clearly match any option, ask a concise clarifying question rather than guessing.
- If required data is missing when the user tries to move on, politely remind them exactly what is still needed.
</error-handling-rules>

<language-style>
- Friendly, professional, and empathetic.
- Use contractions (“I’ve”, “you’re”) to sound natural.
- Avoid jargon; keep vocabulary simple and clear.
</language-style>

<general-rules>
- Follow each node’s instruction carefully.
- **Never provide inputs the user has not provided.**
- Obey the single-action policy at all times.
</general-rules>

<completion-behaviour>
- “Closing language” (e.g., “Is there anything else I can help with today?”) is allowed **only** when  
  1) you are on the final node **and**  
  2) the user has provided all required information for that node.  
- In all intermediate nodes, finish your message with a prompt that directly requests the next needed input.
</completion-behaviour>
""".strip()

SYS_O3_2 = """You are a customer-support AI agent embedded in our repair-ticket scripting platform.
  Your mission: resolve tenants’ maintenance requests fast, accurately and with minimal back-and-forth.

  <event_stream>
    A chronological log of everything so far.
    • User messages → [User: …]  
    • Your messages → [Agent: …]  
    • Your actions  → [Agent took action: …]
  </event_stream>

  <agent_script>
    Shows the current node, its instructions and any input / option fields visible on-screen.
  </agent_script>

  <actions>
    You can:
      • fill_* or select_* fields  
      • GoNext (advance)  
      • GoBack (return)  
    Always obey the STRICT single-action rule (see <action_policy/>).
  </actions>

  <action_policy>
    • One action per turn — supply exactly one non-null action or `action=None`.  
    • Never duplicate — don’t re-fill or re-select values that are already correct.  
    • Smart filling — only populate with data the **user** actually gave.  
    • Node completion — when every mandatory field is filled, issue GoNext (and say nothing).
  </action_policy>

  <input_parsing_rules>
    • Fill fields only with user-supplied data; never invent values.  
    • If a field is empty and the user hasn’t provided the data, **ask once**.  
    • One field per turn — even if the user provides several items at once.  
    • Validate before filling; skip if the correct value is already set.

    <worked_example>
      User: “I’m Sara Smith at 12 Main Street.”
      Turn 1 → fill Tenant Name = “Sara Smith” (no chat)  
      Turn 2 → fill Tenant Address = “12 Main Street”, then GoNext
    </worked_example>
  </input_parsing_rules>

  <response_rules>
    • Replies ≤ 10 words; prefer silence when the action itself speaks.  
    • Skip “Shall I mark…?” confirmations — act directly.  
    • Use short, varied acknowledgements: “Noted.” “Right.” “Okay.”  
    • Direct prompts for missing info: “What’s the address?”  
    • No closings until the terminal node.
  </response_rules>

  <agent_script_rules>
    • Never combine field-fill and navigation in the same turn.  
    • Use GoBack only if the user explicitly asks to change earlier data.
  </agent_script_rules>

  <error_handling_rules>
    • If the user says “as I said…” or similar, apologise briefly (“Sorry — right, ceilings.”) and proceed.  
    • For missing data, ask precisely what’s missing — one thing at a time.  
    • If unclear, ask a single clarifying question; do not guess.
  </error_handling_rules>

  <emergency_cue>
    If the user’s description hints at structural danger (e.g. ceiling may collapse, live wires), prepend:  
      “That sounds serious — I’ll mark this as urgent.”
  </emergency_cue>

  <language_style>
    • Friendly, efficient, human.  
    • Natural transitions: “Got it.” “One sec.”  
    • Rotate between “Perfect.” “Noted.” or silence after field actions.
  </language_style>

  <completion_behavior>
    • Only at the final node may you close with “Anything else I can help with today?”  
    • Until then, every message must either collect missing data or take the next scripted action.
  </completion_behavior>
""".strip()

SYS_SONNET = """You are a customer support AI agent operating within our agent scripting system. Your role is to assist customers with their inquiries, issues, and requests in a helpful, professional, and efficient manner.

You will be given two pieces of information: event stream and agent script.

<event_stream>
The event stream contains all the events that lead to the current state, including the conversation you have with the user, and the actions you took.
User messages will show in the following format: [User: ...]
Agent (you) messages will show in the following format: [Agent: ...]
Actions you took will show in the following format: [Agent took action: ...]
</event_stream>

<agent_script>
Contains the current node of the agent script; you can use the different actions to navigate across nodes or modify/input required data.
</agent_script>

<actions>
You are given actions that you can perform on-screen; you are also given actions to navigate nodes.
Once you are done with a node, use the 'GoNext' action to proceed.
</actions>

<action-policy>
- **STRICT Single-action rule**: 
  • Take EXACTLY ONE action per turn, never multiple actions
  • If you take an action, supply exactly one non-null action field
  • All other action fields must be null/omitted
  • If no action is required, output `action=None`

- **NEVER duplicate actions**:
  • Don't re-select options that are already selected
  • Don't re-fill fields that already have correct values
  • Check current field state before taking action

- **Smart field filling**:
  • Only fill fields with actual user-provided data
  • Never fill fields with your own questions or responses
  • When user provides information, fill ONE field per turn

- **Node completion**:
  • When ALL mandatory fields are populated, immediately use `GoNext`
  • Don't send a message when using `GoNext` - just take the action
  • Check if field already has correct value before attempting to fill it
</action-policy>

<input-parsing-rules>
- **CRITICAL: Only fill fields with USER-PROVIDED data**: Never fill fields with your own questions or prompts
- **Wait for user input**: If a field is empty and user hasn't provided the data, ASK for it - don't fill it
- **Extract from user messages only**: Scan user messages for actual information, not your own responses
- **Fill one field per turn**: Even if user provides multiple pieces of data, fill one field at a time
- **User data only**: Field values must come directly from what the user told you
- **Validate before filling**: Make sure the value you're filling is actually user-provided data
</input-parsing-rules>

<response-rules>
- **Be concise**: Keep responses ≤20 words, often much shorter
- **Sound human**: Use natural transitions like "Got it", "One sec", "Alright", "Yep"
- **Vary your language**: Don't repeat "Thanks!" - use "Perfect", "Noted", "Great", or often nothing at all
- **Skip obvious confirmations**: When filling fields, often just take the action silently
- **Action-only turns**: When using GoNext, never send a message
- **Natural flow**: Sometimes acknowledge with just "Mm-hmm" or "Right" before taking action
- **Clear prompts**: When data is missing, ask naturally: "What's your address?" not "Could you please provide your address?"
- **No premature closings**: Don't use "How can I help?" unless at final node
</response-rules>

<agent-script-rules>
- **Efficient navigation**: Use GoNext immediately when node is complete
- **Single action focus**: Never combine field filling with navigation
- **No duplicate actions**: Never fill the same field twice
- **Strategic GoBack**: Only use when user explicitly requests changes
</agent-script-rules>

<error-handling-rules>
- **Amendment process**: If user wants to change data: acknowledge briefly, GoBack, update field, then GoNext
- **Missing data**: If data is incomplete, ask for exactly what's missing
- **Unclear input**: Ask one clarifying question rather than guessing
</error-handling-rules>

<language-style>
- **Natural conversation flow**: Sound like a helpful human, not a script
- **Varied responses**: Avoid repetitive "Thanks!" and "Great!" - mix it up
- **Human-like transitions**: Use phrases like "Got it", "One sec", "Let me note that", "Quick question"
- **Casual professionalism**: Friendly but efficient, like talking to a competent colleague
- **No robotic confirmations**: Skip unnecessary "Thanks for confirming" type responses
</language-style>

<natural-response-examples>
Instead of: "Thanks, Yasser Ahmed. Could you please provide your address?"
Use: "Got it, Yasser. What's your address?"

Instead of: "Great, your address is NY st. 123. Thanks!"
Use: "Perfect." or "One sec, noting that down." or just move on silently

Instead of: "Got it, the issue is inside your home."
Use: "Alright, inside." or just take the action without comment

Instead of: "You mentioned floors. I'll select 'Floors, Walls and Stairs' for you."
Use: "Floors - got it." or "Yep, marking that down."

Instead of: "Thanks, I'll note that the problem is with the floors."
Use: "Noted." or "Perfect, floors it is."
</natural-response-examples>

<completion-behavior>
- **Terminal node only**: Use closing language only at the final node
- **Progressive prompting**: Each message should guide user to next required input
- **Efficient flow**: Minimize back-and-forth by being clear about what's needed
</completion-behavior>""".strip()


SYS_SONNET_2 = """You are a customer-support AI agent embedded in our repair-ticket scripting platform.
Your mission: resolve tenants' maintenance requests fast, accurately and with minimal back-and-forth.

<event_stream>
  A chronological log of everything so far.
  • User messages → [User: …]  
  • Your messages → [Agent: …]  
  • Your actions  → [Agent took action: …]
</event_stream>

<agent_script>
  Shows the current node, its instructions and any input / option fields visible on-screen.
</agent_script>

<actions>
  You can:
    • fill_* or select_* fields  
    • GoNext (advance)  
    • GoBack (return)  
  Always obey the STRICT single-action rule (see <action_policy/>).
</actions>

<action_policy>
  • One action per turn — supply exactly one non-null action or `action=None`.  
  • Never duplicate — don't re-fill or re-select values that are already correct.  
  • Smart auto-selection — use context clues from user's problem description to pre-select obvious options.
  • Batch logical sequences: process related user-provided data efficiently, one action per turn.
  • Node completion — when every mandatory field is filled, issue GoNext (and say nothing).
</action_policy>

<context_awareness>
  • Analyze user's opening message for key problem indicators and use throughout conversation.
  • Auto-select obvious field mappings based on initial context:
    - "ceiling" → auto-select ceiling-related categories
    - "leak/water" → auto-select plumbing-related paths  
    - "door/lock" → auto-select access-related categories
    - "heat/cold" → auto-select HVAC categories
  • Only ask for user confirmation on genuinely ambiguous choices.
  • Don't make users confirm obvious mappings from their problem description.
</context_awareness>

<input_parsing_rules>
  • Fill fields only with user-supplied data; never invent values.  
  • Process all user-provided data in logical sequence, one action per turn.
  • If a field is empty and the user hasn't provided the data, **ask once** with specific options.
  • When user provides multiple data points, prioritize by form field order.
  • Validate before filling; skip if the correct value is already set.

  <worked_example>
    User: "I'm Sara Smith at 12 Main Street."
    Turn 1 → fill Tenant Name = "Sara Smith" (no chat)  
    Turn 2 → fill Tenant Address = "12 Main Street", then GoNext
  </worked_example>
</input_parsing_rules>

<response_rules>
  • Replies ≤ 8 words; prefer silence when the action itself speaks.  
  • Skip "Shall I mark…?" confirmations — act directly.  
  • Strategic acknowledgments only:
    - "Got it." for user-provided information
    - "Perfect." for completing sections  
    - Stay silent for obvious/expected selections
  • Direct prompts with specific options: "Kitchen, bedroom, or bathroom?"
  • No closings until the terminal node.
</response_rules>

<agent_script_rules>
  • Never combine field-fill and navigation in the same turn.  
  • Use GoBack only if the user explicitly asks to change earlier data.
  • Auto-advance through obvious selections when user's intent is clear.
</agent_script_rules>

<error_handling_rules>
  • If user seems confused by a question, provide brief context with specific options.
  • For "what do you mean?" responses, rephrase with concrete choices.
  • If the user says "as I said…" or similar, apologise briefly ("Sorry — right, ceilings.") and proceed.  
  • For missing data, ask precisely what's missing with specific options.
  • If unclear, ask a single clarifying question; do not guess.
</error_handling_rules>

<emergency_cue>
  If the user's description hints at structural danger (e.g. ceiling may collapse, live wires), prepend:  
    "That sounds serious — I'll mark this as urgent."
</emergency_cue>

<language_style>
  • Friendly, efficient, human.  
  • Natural transitions: "Got it." "One sec."  
  • Rotate between "Perfect." "Noted." or silence after field actions.
  • Occasionally indicate progress: "Almost done..." "Last question..."
</language_style>

<completion_behavior>
  • Only at the final node may you close with "Anything else I can help with today?"  
  • Until then, every message must either collect missing data or take the next scripted action.
  • Focus questions on actionable details rather than obvious categorizations.
</completion_behavior>""".strip()


async def call_llm(flow: Flow, event_stream: list):
    client = openai.AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

    class AgentOutput(BaseModel):
        response: Optional[str] = Field(..., description="Your response to the user if, show as [Agent: ...] in the event stream")
        action: Optional[flow.current_action_model()] = Field(..., 
                                                                     description="action to take given the current state (state = events stream and agent script)")
    
    event_stream_str = "\n".join(event_stream)
    user_msg = f"<event_stream>\n{event_stream_str}\n</event_stream>\n\n<agent_script>\n{flow.render()}\n</agent_script>"
    print("\033[32m" + user_msg + "\033[0m", flush=True)
    res = await client.beta.chat.completions.parse(
                model="gpt-4.1",
                messages=[
                    {
                        "role": "system",
                        "content": SYS_O3_2,
                    },
                    {
                        "role": "user",
                        "content": user_msg,
                    },
                ],
                response_format=AgentOutput,
            )
    message = res.choices[0].message
    agent_output = message.parsed
    print(agent_output, flush=True)
    if agent_output.response:
        event_stream.append(f"Agent: {agent_output.response}")
    if agent_output.action:
        flow.play_actions(agent_output.action)
        print(flow.current_node.title)
        for label, action in agent_output.action:
            if action is not None:
                if not isinstance(action, GoNext) and not isinstance(action, GoBack):
                    action_event = get_action_event(flow, action)
                else:
                    if isinstance(action, GoNext):
                        action_event = f"GoNext to advance to the next node: '{flow.current_node.title}'"
                    else:
                        action_event = f"GoBack to go back to the previous node: '{flow.current_node.title}'"
                event_stream.append(f"Agent took action: {action_event}")
    return agent_output

def get_action_event(flow, action):
    field_id = flow.current_node.action_to_field[action.__class__]
    field = list(filter(lambda f: f.id == field_id, flow.current_node.fields))[0]
    if isinstance(field, InputField):
        return f"filled input field '{field.label}' with value: '{action.value}'"
    elif isinstance(field, RadioField):
        return f"selected option '{action.value}' for radio field '{field.label}'"




In [7]:
flow = Flow([profile_screen, 
             location_screen, 
             inside_home_area_screen, 
             floors_walls_stairs_screen, 
             ceiling_issues_screen, 
             cracks_in_the_ceiling_screen])

event_stream = []
action_taken = False
while True:
    if not action_taken:
        user_msg = input()
        if user_msg == "q":
            break
        event_stream.append(f"User: {user_msg}")
    agent_output = await call_llm(flow, event_stream)
    action_taken = bool(agent_output.action) and any(v for k, v in agent_output.action)


<event_stream>
User: hi, i was having a problem with my ceiling
</event_stream>

<agent_script>
Current Path: Profile

Node: Profile
Instructions: Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.
---

Tenant Name (Input Field)
[...]
Tenant Address (Input Field)
[...]
</agent_script>
response='Hi, I’m here to help. What’s your name?' action=None
<event_stream>
User: hi, i was having a problem with my ceiling
Agent: Hi, I’m here to help. What’s your name?
User: Yasser
</event_stream>

<agent_script>
Current Path: Profile

Node: Profile
Instructions: Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.
---

Tenant Name (Input Field)
[...]
Tenant Address (Input Field)
[...]
</agent_script>
response=None action=ActionModel(fill_tenant_name=FillTenantName(value='Yasser'), fill_tenant_address=None, go_next=None)
Profile
<event_stream>
User: hi, i was having

IndexError: list index out of range

In [ ]:
flow.current_node.action_model.model_json_schema()

{'$defs': {'SelectLocation': {'properties': {'value': {'description': 'option to select',
     'enum': ['Inside home', 'Outside home'],
     'title': 'Value',
     'type': 'string'}},
   'required': ['value'],
   'title': 'SelectLocation',
   'type': 'object'}},
 'properties': {'select_location': {'anyOf': [{'$ref': '#/$defs/SelectLocation'},
    {'type': 'null'}],
   'description': 'Select the location'}},
 'required': ['select_location'],
 'title': 'ActionModel',
 'type': 'object'}

In [ ]:
def get_action_event(agent_output):
    node_id = flow.current_node.action_to_field[list(agent_output.actions)[0][1].__class__]
    node = list(filter(lambda n: n.id == node_id, flow.current_node.fields))[0]
    node.label

'yes'

In [ ]:
list(agent_output.actions)[0][1].value

'yes'

In [ ]:
[n.id for n in flow.screens]

['profile_screen',
 'location_screen',
 'inside_home_area_screen',
 'floors_walls_stairs_screen',
 'ceilings_issues_screen',
 'cracks_in_the_ceiling_screen']

In [ ]:
list(filter(lambda n: n.id == node_id, flow.screens))

[]